# Example 1: Convex Sets and Optimization (Interactive)

This notebook provides two interactive parts:

1. **Part 1**: Build sets (hyperplane, halfspace, ellipsoid), combine them via intersection or union, and check convexity guarantees from lecture rules.
2. **Part 2**: Build and combine sets again, then solve a linear or quadratic optimization problem on top of them, with automatic case classification.

## Quick Start

Run cells top to bottom. If widgets are not responsive, run the `%pip install ...` cell and restart the kernel.

In [ ]:
# Optional dependency install (uncomment if needed)
# %pip install numpy matplotlib ipywidgets cvxpy

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
import cvxpy as cp

from IPython.display import display, Markdown

# Prefer interactive backend in VS Code/Jupyter, fallback silently if unavailable.
try:
    get_ipython().run_line_magic('matplotlib', 'widget')
except Exception:
    pass

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['axes.grid'] = True
plt.rcParams['font.size'] = 11

# Shared plotting domain and numerical tolerances.
X_MIN, X_MAX = -6.0, 6.0
Y_MIN, Y_MAX = -6.0, 6.0
GRID_N = 280
EPS_MASK = 0.04
EPS_ACTIVE = 1e-3
EPS_OPT = 3e-2


def rotation_matrix(theta_rad):
    c, s = np.cos(theta_rad), np.sin(theta_rad)
    return np.array([[c, -s], [s, c]])


def meshgrid_domain():
    xs = np.linspace(X_MIN, X_MAX, GRID_N)
    ys = np.linspace(Y_MIN, Y_MAX, GRID_N)
    return np.meshgrid(xs, ys)


def build_set_controls(title='Set Controls'):
    use_hyperplane = widgets.Checkbox(value=True, description='Enable hyperplane')
    hp_a1 = widgets.FloatSlider(value=1.0, min=-3.0, max=3.0, step=0.1, description='hp a1', continuous_update=False)
    hp_a2 = widgets.FloatSlider(value=1.0, min=-3.0, max=3.0, step=0.1, description='hp a2', continuous_update=False)
    hp_b = widgets.FloatSlider(value=0.0, min=-6.0, max=6.0, step=0.1, description='hp b', continuous_update=False)

    use_halfspace = widgets.Checkbox(value=True, description='Enable halfspace')
    hs_a1 = widgets.FloatSlider(value=1.0, min=-3.0, max=3.0, step=0.1, description='hs a1', continuous_update=False)
    hs_a2 = widgets.FloatSlider(value=-0.5, min=-3.0, max=3.0, step=0.1, description='hs a2', continuous_update=False)
    hs_b = widgets.FloatSlider(value=2.0, min=-6.0, max=6.0, step=0.1, description='hs b', continuous_update=False)

    use_ellipsoid = widgets.Checkbox(value=True, description='Enable ellipsoid')
    el_cx = widgets.FloatSlider(value=0.0, min=-5.0, max=5.0, step=0.1, description='el cx', continuous_update=False)
    el_cy = widgets.FloatSlider(value=0.0, min=-5.0, max=5.0, step=0.1, description='el cy', continuous_update=False)
    el_rx = widgets.FloatSlider(value=2.5, min=0.5, max=5.0, step=0.1, description='el rx', continuous_update=False)
    el_ry = widgets.FloatSlider(value=1.5, min=0.5, max=5.0, step=0.1, description='el ry', continuous_update=False)
    el_angle = widgets.FloatSlider(value=20.0, min=-90.0, max=90.0, step=1.0, description='el angle', continuous_update=False)

    operator = widgets.Dropdown(options=['Intersection', 'Union'], value='Intersection', description='Combine')

    header = widgets.HTML(f"<h4 style='margin:0'>{title}</h4>")

    panel = widgets.VBox([
        header,
        operator,
        widgets.HTML('<b>Hyperplane: a^T x = b</b>'), use_hyperplane, hp_a1, hp_a2, hp_b,
        widgets.HTML('<b>Halfspace: a^T x <= b</b>'), use_halfspace, hs_a1, hs_a2, hs_b,
        widgets.HTML('<b>Ellipsoid: (x-c)^T A^-1 (x-c) <= 1</b>'), use_ellipsoid, el_cx, el_cy, el_rx, el_ry, el_angle,
    ], layout=widgets.Layout(width='360px'))

    return {
        'panel': panel,
        'operator': operator,
        'use_hyperplane': use_hyperplane,
        'hp_a1': hp_a1,
        'hp_a2': hp_a2,
        'hp_b': hp_b,
        'use_halfspace': use_halfspace,
        'hs_a1': hs_a1,
        'hs_a2': hs_a2,
        'hs_b': hs_b,
        'use_ellipsoid': use_ellipsoid,
        'el_cx': el_cx,
        'el_cy': el_cy,
        'el_rx': el_rx,
        'el_ry': el_ry,
        'el_angle': el_angle,
    }


def extract_specs(ctrl):
    specs = []

    if ctrl['use_hyperplane'].value:
        a = np.array([ctrl['hp_a1'].value, ctrl['hp_a2'].value], dtype=float)
        if np.linalg.norm(a) > 1e-9:
            specs.append({'type': 'hyperplane', 'a': a, 'b': float(ctrl['hp_b'].value), 'convex': True})

    if ctrl['use_halfspace'].value:
        a = np.array([ctrl['hs_a1'].value, ctrl['hs_a2'].value], dtype=float)
        if np.linalg.norm(a) > 1e-9:
            specs.append({'type': 'halfspace', 'a': a, 'b': float(ctrl['hs_b'].value), 'convex': True})

    if ctrl['use_ellipsoid'].value:
        cx = float(ctrl['el_cx'].value)
        cy = float(ctrl['el_cy'].value)
        rx = float(ctrl['el_rx'].value)
        ry = float(ctrl['el_ry'].value)
        th = np.deg2rad(float(ctrl['el_angle'].value))
        R = rotation_matrix(th)
        A = R @ np.diag([rx**2, ry**2]) @ R.T
        specs.append({'type': 'ellipsoid', 'c': np.array([cx, cy]), 'A': A, 'convex': True})

    return specs


def set_mask(spec, X, Y):
    if spec['type'] == 'hyperplane':
        Z = spec['a'][0] * X + spec['a'][1] * Y - spec['b']
        return np.abs(Z) <= EPS_MASK

    if spec['type'] == 'halfspace':
        Z = spec['a'][0] * X + spec['a'][1] * Y - spec['b']
        return Z <= 0.0

    if spec['type'] == 'ellipsoid':
        c = spec['c']
        A_inv = np.linalg.inv(spec['A'])
        DX = X - c[0]
        DY = Y - c[1]
        V = (
            A_inv[0, 0] * DX**2
            + 2.0 * A_inv[0, 1] * DX * DY
            + A_inv[1, 1] * DY**2
        )
        return V <= 1.0

    return np.zeros_like(X, dtype=bool)


def combine_masks(masks, operator):
    if len(masks) == 0:
        return np.zeros_like(meshgrid_domain()[0], dtype=bool)
    out = masks[0].copy()
    if operator == 'Intersection':
        for m in masks[1:]:
            out &= m
    else:
        for m in masks[1:]:
            out |= m
    return out


def convexity_guarantee(operator, n_sets):
    if n_sets == 0:
        return 'No sets selected', 'Select at least one set.'

    if n_sets == 1:
        return 'Always convex', 'A single selected primitive (hyperplane, halfspace, ellipsoid) is convex.'

    if operator == 'Intersection':
        return 'Always convex', 'Intersection of convex sets is convex.'

    return 'Not always convex', 'Union of convex sets is not guaranteed convex in general.'


def draw_set_boundaries(ax, specs):
    x_line = np.linspace(X_MIN, X_MAX, 500)

    for spec in specs:
        if spec['type'] == 'hyperplane':
            a1, a2 = spec['a']
            b = spec['b']
            if abs(a2) > 1e-9:
                y = (b - a1 * x_line) / a2
                ax.plot(x_line, y, color='black', lw=2.0, label='Hyperplane')
            else:
                x0 = b / a1
                ax.axvline(x=x0, color='black', lw=2.0, label='Hyperplane')

        elif spec['type'] == 'halfspace':
            a1, a2 = spec['a']
            b = spec['b']
            if abs(a2) > 1e-9:
                y = (b - a1 * x_line) / a2
                ax.plot(x_line, y, color='#1f77b4', lw=1.8, ls='--', label='Halfspace boundary')
            else:
                x0 = b / a1
                ax.axvline(x=x0, color='#1f77b4', lw=1.8, ls='--', label='Halfspace boundary')

        elif spec['type'] == 'ellipsoid':
            c = spec['c']
            A = spec['A']
            w, V = np.linalg.eigh(A)
            t = np.linspace(0, 2 * np.pi, 400)
            circ = np.vstack([np.cos(t), np.sin(t)])
            pts = c.reshape(2, 1) + V @ np.diag(np.sqrt(w)) @ circ
            ax.plot(pts[0, :], pts[1, :], color='#d62728', lw=2.0, label='Ellipsoid')

    ax.set_xlim(X_MIN, X_MAX)
    ax.set_ylim(Y_MIN, Y_MAX)
    ax.set_xlabel('x1')
    ax.set_ylabel('x2')
    ax.set_aspect('equal', adjustable='box')

## Part 1: Build Sets and Check Convexity

Use the controls to enable any combination of:
- Hyperplane
- Halfspace
- Ellipsoid

Then select `Intersection` or `Union` as the combine operator.

The notebook reports whether convexity is **guaranteed** by lecture rules.

In [ ]:
part1_ctrl = build_set_controls('Part 1 Controls')
part1_out = widgets.Output()


def update_part1(_=None):
    specs = extract_specs(part1_ctrl)
    op = part1_ctrl['operator'].value

    X, Y = meshgrid_domain()
    masks = [set_mask(s, X, Y) for s in specs]

    part1_out.clear_output(wait=True)
    with part1_out:
        if hasattr(update_part1, '_fig') and update_part1._fig is not None:
            plt.close(update_part1._fig)
        fig, ax = plt.subplots(1, 1, figsize=(8.5, 7.0))
        update_part1._fig = fig

        if len(masks) > 0:
            combined = combine_masks(masks, op)
            ax.contourf(X, Y, combined.astype(float), levels=[0.5, 1.5], colors=['#a6cee3'], alpha=0.45)

        draw_set_boundaries(ax, specs)

        status, explanation = convexity_guarantee(op, len(specs))
        color = '#2ca02c' if status == 'Always convex' else '#d62728'
        if status == 'No sets selected':
            color = '#8c564b'

        ax.set_title(f'Combined Set ({op})')
        ax.text(
            0.02,
            0.98,
            f'Convexity guarantee: {status}\n{explanation}',
            transform=ax.transAxes,
            va='top',
            ha='left',
            bbox=dict(facecolor='white', alpha=0.9, edgecolor=color, linewidth=2),
            color=color,
        )

        # Deduplicate legend labels.
        handles, labels = ax.get_legend_handles_labels()
        seen = set()
        filt_h, filt_l = [], []
        for h, l in zip(handles, labels):
            if l not in seen:
                seen.add(l)
                filt_h.append(h)
                filt_l.append(l)
        if len(filt_h) > 0:
            ax.legend(filt_h, filt_l, loc='lower right')

        plt.show()


for w in part1_ctrl.values():
    if isinstance(w, widgets.Widget) and hasattr(w, 'observe'):
        w.observe(update_part1, names='value')


display(widgets.HBox([part1_ctrl['panel'], part1_out]))
update_part1()

## Part 2: Optimization on Combined Sets (LP / QP)

Use a second, independent set-builder and choose an optimizer:
- **LP**: minimize $c^T x$
- **QP**: minimize $\frac{1}{2}x^T Hx + q^T x$

The notebook classifies:
- LP: Case 1 (unbounded), Case 2 (bounded unique optimum), Case 3 (bounded multiple optima)
- QP: Case 1 (optimizer strictly inside feasible polyhedron), Case 2 (optimizer on boundary)

In [ ]:
def build_opt_controls():
    mode = widgets.Dropdown(options=['LP', 'QP'], value='LP', description='Optimizer')

    c1 = widgets.FloatSlider(value=-1.0, min=-3.0, max=3.0, step=0.1, description='c1', continuous_update=False)
    c2 = widgets.FloatSlider(value=-1.0, min=-3.0, max=3.0, step=0.1, description='c2', continuous_update=False)

    h11 = widgets.FloatSlider(value=1.6, min=0.2, max=4.0, step=0.1, description='H11', continuous_update=False)
    h22 = widgets.FloatSlider(value=1.1, min=0.2, max=4.0, step=0.1, description='H22', continuous_update=False)
    h12 = widgets.FloatSlider(value=0.2, min=-1.5, max=1.5, step=0.05, description='H12', continuous_update=False)
    q1 = widgets.FloatSlider(value=-1.2, min=-4.0, max=4.0, step=0.1, description='q1', continuous_update=False)
    q2 = widgets.FloatSlider(value=-0.8, min=-4.0, max=4.0, step=0.1, description='q2', continuous_update=False)

    note = widgets.HTML(
        '<small><b>Note:</b> LP/QP case labels are exact for convex intersection models solved via cvxpy. '
        'For union or unsupported configurations, the notebook uses a sampled approximation.</small>'
    )

    panel = widgets.VBox([
        widgets.HTML('<h4 style="margin:0">Part 2 Optimization Controls</h4>'),
        mode,
        widgets.HTML('<b>LP objective</b>  minimize  c1*x1 + c2*x2'),
        c1, c2,
        widgets.HTML('<b>QP objective</b>  minimize 0.5*x^T H x + q^T x'),
        h11, h22, h12, q1, q2,
        note,
    ], layout=widgets.Layout(width='360px'))

    return {
        'panel': panel,
        'mode': mode,
        'c1': c1,
        'c2': c2,
        'h11': h11,
        'h22': h22,
        'h12': h12,
        'q1': q1,
        'q2': q2,
    }


def cvx_constraints_for_specs(xvar, specs, operator):
    if operator != 'Intersection':
        raise ValueError('Only intersection is convex-compliant for cvxpy constraints.')

    cons = []
    for s in specs:
        if s['type'] == 'halfspace':
            cons.append(s['a'] @ xvar <= s['b'])
        elif s['type'] == 'hyperplane':
            cons.append(s['a'] @ xvar == s['b'])
        elif s['type'] == 'ellipsoid':
            A_inv = np.linalg.inv(s['A'])
            cons.append(cp.quad_form(xvar - s['c'], A_inv) <= 1.0)
    return cons


def sample_feasible_points(specs, operator):
    X, Y = meshgrid_domain()
    masks = [set_mask(s, X, Y) for s in specs]
    if len(masks) == 0:
        return X, Y, np.zeros_like(X, dtype=bool)
    combined = combine_masks(masks, operator)
    return X, Y, combined


def classify_lp_from_samples(values, feasible_mask):
    if np.sum(feasible_mask) == 0:
        return 'Infeasible', None

    vf = values[feasible_mask]
    vmin = np.min(vf)
    span = max(np.max(vf) - np.min(vf), 1e-9)
    near = np.abs(vf - vmin) <= max(EPS_OPT, 0.01 * span)

    if np.sum(near) > 40:
        return 'Case 3: Bounded with multiple optima', vmin
    return 'Case 2: Bounded with unique optimum', vmin


def classify_qp_boundary_from_point(x_star, specs):
    active = False
    for s in specs:
        if s['type'] == 'halfspace':
            slack = s['b'] - s['a'] @ x_star
            if abs(slack) <= 5e-2:
                active = True
        elif s['type'] == 'hyperplane':
            active = True
        elif s['type'] == 'ellipsoid':
            A_inv = np.linalg.inv(s['A'])
            v = (x_star - s['c']).T @ A_inv @ (x_star - s['c'])
            if abs(v - 1.0) <= 5e-2:
                active = True
    if active:
        return 'Case 2: Optimizer on boundary'
    return 'Case 1: Optimizer strictly inside feasible polyhedron'


def solve_opt_problem(specs, operator, opt_ctrl):
    mode = opt_ctrl['mode'].value

    # Grid data for plotting and fallback classification.
    X, Y, feasible_mask = sample_feasible_points(specs, operator)

    if mode == 'LP':
        c = np.array([opt_ctrl['c1'].value, opt_ctrl['c2'].value], dtype=float)
        Z = c[0] * X + c[1] * Y

        # Try exact convex solve where possible.
        x_star = None
        label = 'Case unknown'
        solver_note = 'sampled'

        if len(specs) > 0 and operator == 'Intersection':
            try:
                x = cp.Variable(2)
                prob = cp.Problem(cp.Minimize(c @ x), cvx_constraints_for_specs(x, specs, operator))
                prob.solve(solver=cp.SCS, verbose=False)
                if prob.status in ('unbounded', 'unbounded_inaccurate'):
                    label = 'Case 1: Unbounded'
                    solver_note = 'cvxpy'
                elif prob.status in ('infeasible', 'infeasible_inaccurate'):
                    label = 'Infeasible'
                    solver_note = 'cvxpy'
                elif prob.status in ('optimal', 'optimal_inaccurate'):
                    x_star = np.array(x.value).reshape(-1)
                    label_s, _ = classify_lp_from_samples(Z, feasible_mask)
                    label = label_s
                    solver_note = 'cvxpy + sampled multiplicity check'
                else:
                    label, _ = classify_lp_from_samples(Z, feasible_mask)
            except Exception:
                label, _ = classify_lp_from_samples(Z, feasible_mask)
        else:
            label, _ = classify_lp_from_samples(Z, feasible_mask)

        if x_star is None and np.sum(feasible_mask) > 0:
            idx = np.argmin(np.where(feasible_mask, Z, np.inf))
            i, j = np.unravel_index(idx, Z.shape)
            x_star = np.array([X[i, j], Y[i, j]])

        return {
            'mode': 'LP',
            'X': X,
            'Y': Y,
            'feasible': feasible_mask,
            'Z': Z,
            'x_star': x_star,
            'label': label,
            'solver_note': solver_note,
        }

    # QP branch
    h11v = float(opt_ctrl['h11'].value)
    h22v = float(opt_ctrl['h22'].value)
    h12_raw = float(opt_ctrl['h12'].value)
    # Keep H positive definite by clipping off-diagonal magnitude.
    h12_lim = 0.95 * np.sqrt(h11v * h22v)
    h12v = np.clip(h12_raw, -h12_lim, h12_lim)

    H = np.array([[h11v, h12v], [h12v, h22v]], dtype=float)
    q = np.array([opt_ctrl['q1'].value, opt_ctrl['q2'].value], dtype=float)

    Z = 0.5 * (H[0, 0] * X**2 + 2 * H[0, 1] * X * Y + H[1, 1] * Y**2) + q[0] * X + q[1] * Y

    x_star = None
    label = 'Case unknown'
    solver_note = 'sampled'

    if len(specs) > 0 and operator == 'Intersection':
        try:
            x = cp.Variable(2)
            obj = 0.5 * cp.quad_form(x, H) + q @ x
            prob = cp.Problem(cp.Minimize(obj), cvx_constraints_for_specs(x, specs, operator))
            prob.solve(solver=cp.SCS, verbose=False)

            if prob.status in ('infeasible', 'infeasible_inaccurate'):
                label = 'Infeasible'
                solver_note = 'cvxpy'
            elif prob.status in ('optimal', 'optimal_inaccurate'):
                x_star = np.array(x.value).reshape(-1)
                label = classify_qp_boundary_from_point(x_star, specs)
                solver_note = 'cvxpy'
            else:
                if np.sum(feasible_mask) > 0:
                    idx = np.argmin(np.where(feasible_mask, Z, np.inf))
                    i, j = np.unravel_index(idx, Z.shape)
                    x_star = np.array([X[i, j], Y[i, j]])
                    label = classify_qp_boundary_from_point(x_star, specs)
        except Exception:
            if np.sum(feasible_mask) > 0:
                idx = np.argmin(np.where(feasible_mask, Z, np.inf))
                i, j = np.unravel_index(idx, Z.shape)
                x_star = np.array([X[i, j], Y[i, j]])
                label = classify_qp_boundary_from_point(x_star, specs)
    else:
        if np.sum(feasible_mask) > 0:
            idx = np.argmin(np.where(feasible_mask, Z, np.inf))
            i, j = np.unravel_index(idx, Z.shape)
            x_star = np.array([X[i, j], Y[i, j]])
            label = classify_qp_boundary_from_point(x_star, specs)

    return {
        'mode': 'QP',
        'X': X,
        'Y': Y,
        'feasible': feasible_mask,
        'Z': Z,
        'x_star': x_star,
        'label': label,
        'solver_note': solver_note,
    }


part2_set_ctrl = build_set_controls('Part 2 Set Controls')
part2_opt_ctrl = build_opt_controls()
part2_out = widgets.Output()


def update_part2(_=None):
    specs = extract_specs(part2_set_ctrl)
    op = part2_set_ctrl['operator'].value

    part2_out.clear_output(wait=True)
    with part2_out:
        if hasattr(update_part2, '_fig') and update_part2._fig is not None:
            plt.close(update_part2._fig)
        result = solve_opt_problem(specs, op, part2_opt_ctrl)

        X = result['X']
        Y = result['Y']
        feasible = result['feasible']
        Z = result['Z']
        x_star = result['x_star']

        fig, ax = plt.subplots(1, 1, figsize=(9.2, 7.2))
        update_part2._fig = fig

        if np.sum(feasible) > 0:
            Zm = np.where(feasible, Z, np.nan)
            im = ax.pcolormesh(X, Y, Zm, shading='auto', cmap='RdYlBu_r', alpha=0.9)
            fig.colorbar(im, ax=ax, label='Objective value (small = better)')
            cs = ax.contour(X, Y, Zm, levels=10, colors='black', linewidths=0.8, alpha=0.65)
            ax.clabel(cs, inline=True, fontsize=8, fmt='%.2f')
            ax.contourf(X, Y, feasible.astype(float), levels=[0.5, 1.5], colors=['#b3e2cd'], alpha=0.18)
        else:
            ax.text(0.5, 0.5, 'No feasible points in current view', transform=ax.transAxes, ha='center', va='center')

        draw_set_boundaries(ax, specs)

        if x_star is not None and np.all(np.isfinite(x_star)):
            ax.scatter([x_star[0]], [x_star[1]], s=140, marker='*', color='#ff7f00', edgecolor='black', zorder=8, label='Optimizer')

        title = f"{result['mode']} over combined set ({op})"
        ax.set_title(title)

        note_color = '#1b9e77'
        if 'Case 1' in result['label']:
            note_color = '#7570b3'
        elif 'Case 2' in result['label']:
            note_color = '#1b9e77'
        elif 'Case 3' in result['label']:
            note_color = '#d95f02'
        elif 'Infeasible' in result['label']:
            note_color = '#e7298a'

        ax.text(
            0.02,
            0.98,
            f"Classification: {result['label']}\nSolve mode: {result['solver_note']}",
            transform=ax.transAxes,
            va='top',
            ha='left',
            bbox=dict(facecolor='white', alpha=0.92, edgecolor=note_color, linewidth=2),
            color=note_color,
        )

        handles, labels = ax.get_legend_handles_labels()
        seen = set()
        h2, l2 = [], []
        for h, l in zip(handles, labels):
            if l not in seen:
                seen.add(l)
                h2.append(h)
                l2.append(l)
        if len(h2) > 0:
            ax.legend(h2, l2, loc='lower right')

        plt.show()


for w in list(part2_set_ctrl.values()) + list(part2_opt_ctrl.values()):
    if isinstance(w, widgets.Widget) and hasattr(w, 'observe'):
        w.observe(update_part2, names='value')


left_panel = widgets.VBox([part2_set_ctrl['panel'], widgets.HTML('<hr style="margin:6px 0">'), part2_opt_ctrl['panel']])
display(widgets.HBox([left_panel, part2_out]))
update_part2()